# Breakpoint Scan Demo

This notebook demonstrates the `breakpoint_scan()` operation, which splits sequences at specified breakpoint positions into multiple segments.


In [2]:
from poolparty import (
    breakpoint_scan,
    from_seqs,
    concatenate,
    MultiPool,
    reset_op_id_counter,
)
reset_op_id_counter()


## 1. Basic Usage: Single Breakpoint

Split a sequence into 2 segments with 1 breakpoint.


In [5]:
# Split sequence at position 4
multi = breakpoint_scan("ACGTCGCG", num_breakpoints=1, positions=[4], mode='sequential')

print(f"Type: {type(multi)}")
print(f"Number of outputs: {len(multi)}")


Type: <class 'poolparty.pool.MultiPool'>
Number of outputs: 2


In [6]:
# Unpack the segments
left, right = multi

print(f"Left segment:  '{left.seq}'")
print(f"Right segment: '{right.seq}'")
print(f"Reconstructed: '{left.seq + right.seq}'")


Left segment:  'ACGT'
Right segment: 'CGCG'
Reconstructed: 'ACGTCGCG'


## 2. Multiple Breakpoints

With `num_breakpoints=2`, we get 3 segments.


In [7]:
reset_op_id_counter()

multi = breakpoint_scan(
    "AAAABBBBCCCC",
    num_breakpoints=2,
    positions=[4, 8],
    mode='sequential'
)

left, middle, right = multi

print(f"Left:   '{left.seq}'")
print(f"Middle: '{middle.seq}'")
print(f"Right:  '{right.seq}'")


Left:   'AAAA'
Middle: 'BBBB'
Right:  'CCCC'


## 3. Sequential Mode: Iterating Over Combinations

In sequential mode, `breakpoint_scan` iterates over all C(n, k) combinations of breakpoint positions.


In [8]:
from math import comb

reset_op_id_counter()

# 3 possible positions, choosing 1 breakpoint = C(3,1) = 3 states
multi = breakpoint_scan(
    "ABCDEFGH",
    num_breakpoints=1,
    positions=[2, 4, 6],  # 3 possible breakpoint positions
    mode='sequential'
)

print(f"Number of states: {multi.operation.num_states}")
print(f"Expected C(3,1): {comb(3, 1)}")


Number of states: 3
Expected C(3,1): 3


In [9]:
# Generate all 3 states
left, right = multi

df_left = left.generate_library(num_seqs=3, init_state=0)
df_right = right.generate_library(num_seqs=3, init_state=0)

print("All breakpoint combinations:")
for i in range(3):
    print(f"  State {i}: left='{df_left.iloc[i]['seq']}', right='{df_right.iloc[i]['seq']}'")


All breakpoint combinations:
  State 0: left='AB', right='CDEFGH'
  State 1: left='ABCD', right='EFGH'
  State 2: left='ABCDEF', right='GH'


## 4. Using Segments in Operations

Segments can be used as inputs to other operations like `concatenate`.


In [10]:
reset_op_id_counter()

# Split sequence
multi = breakpoint_scan("ACGTACGT", num_breakpoints=1, positions=[4], mode='sequential')
left, right = multi

# Insert a spacer between segments
spacer = from_seqs(["----"])
modified = left + spacer + right

print(f"Original:    'ACGTACGT'")
print(f"With spacer: '{modified.seq}'")


Original:    'ACGTACGT'
With spacer: 'ACGT----ACGT'


In [11]:
# Replace the middle segment entirely
reset_op_id_counter()

multi = breakpoint_scan("AAAABBBBCCCC", num_breakpoints=2, positions=[4, 8], mode='sequential')
left, middle, right = multi

# Swap middle with something else
new_middle = from_seqs(["XXXX"])
modified = left + new_middle + right

print(f"Original: 'AAAABBBBCCCC'")
print(f"Modified: '{modified.seq}'")


Original: 'AAAABBBBCCCC'
Modified: 'AAAAXXXXCCCC'


## Summary

`breakpoint_scan()` is a multi-output operation that:

1. **Splits sequences** at specified breakpoint positions
2. **Returns a `MultiPool`** containing `num_breakpoints + 1` segment Pools
3. **Supports sequential mode** iterating over C(n,k) breakpoint combinations
4. **Supports random mode** for stochastic breakpoint selection
5. **Segments can be used** in further operations like `concatenate`

```python
# Basic usage
left, right = breakpoint_scan(seq, num_breakpoints=1, positions=[pos])

# Multiple breakpoints
seg1, seg2, seg3 = breakpoint_scan(seq, num_breakpoints=2)

# Modify and reconstruct
modified = left + some_new_sequence + right
```
